## Background

This notebook fetches historical stock price data from the [AlphaVantage API](https://www.alphavantage.co/documentation/), and then sends a report about that stock as an email with attachments.



## Objectives

Our code will implement solutions to the following example challenges.

**Challenge A (Latest Close)**

For a given stock...

What is the most recent adjusted closing price? And the corresponding date? Display the price formatted as USD, with dollar sign and two decimal places.


**Challenge B (Recent Stats)**

What is the average, median, min, and max adjusted closing price (over the latest 100 available days only)? Display the prices formatted as USD, with dollar sign and two decimal places.


**Challenge C (Dataviz)**

Plot a line chart of adjusted closing prices over time (all time). Include title and axis labels. The symbol should be in the title.




**Challenge D (Email Report)**

Send an email to let someone know what the latest closing price is. Example email content (although the real values may differ):

"The latest adjusted closing price for NFLX is $628.35 as of 2024-07-31."


**Challenge E (Email Report w/ Attachments)**

Send the same email from Challenge D, but also include some attachments:

  1. Attach a CSV file of historical price data.
  2. Attach a PNG file displaying a dataviz from Challenge C.



## Setup

### Plotting Dependencies

In [1]:
# Fixes:
# > Warning: You have Plotly version 5.24.1, which is not compatible with this version of Kaleido (1.1.0).
# > This means that static image generation (e.g. `fig.write_image()`) will not work.
# > Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.
!pip uninstall plotly -y
!pip install plotly>=6.1.1

Found existing installation: plotly 6.3.1
Uninstalling plotly-6.3.1:
  Successfully uninstalled plotly-6.3.1


In [2]:
!pip show plotly

Name: plotly
Version: 6.3.1
Summary: An open-source interactive data visualization library for Python
Home-page: https://plotly.com/python/
Author: 
Author-email: Chris P <chris@plot.ly>
License: MIT License

Copyright (c) 2016-2024 Plotly Technologies Inc.

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOS

For email attachments (run and then restart the runtime)

In [21]:
#%%capture
# NOTE: run this cell, then "Runtime" > "Restart the session",
!pip uninstall kaleido -y
!pip install -U kaleido==1.1.0

Found existing installation: kaleido 1.1.0
Uninstalling kaleido-1.1.0:
  Successfully uninstalled kaleido-1.1.0
  Using cached kaleido-1.1.0-py3-none-any.whl.metadata (5.6 kB)
Using cached kaleido-1.1.0-py3-none-any.whl (66 kB)


In [17]:
# RuntimeError: Kaleido requires Google Chrome to be installed.
# Either download and install Chrome yourself following Google's instructions for your operating system,
# or install it from your terminal by running:
!plotly_get_chrome -y


Plotly will install a copy of Google Chrome to be used for generating static images of plots.
Chrome will be installed at: None
Do you want to proceed? [y/n] y
Installing Chrome for Plotly...
Chrome installed successfully.
The Chrome executable is now located at: /usr/local/lib/python3.12/dist-packages/choreographer/cli/browser_exe/chrome-linux64/chrome


In [3]:

# import plotly.io as pio

# pio.renderers.default = 'png'

### SendGrid API Setup

In this implementation, we will send emails using the sendgrid service (**see email sending demo notebook**).

Prerequisites: setup a [sendgrid account](https://signup.sendgrid.com/
 ) and single-sender address (i.e. `SENDGRID_API_KEY`), and obtain an api key (i.e. `SENDGRID_API_KEY`). Set these values as notebook secrets.

Installing a specific version of the package:

In [3]:
%%capture
!pip install sendgrid==6.5.0

Accessing notebook secrets:

In [4]:
from google.colab import userdata

SENDGRID_API_KEY = userdata.get("SENDGRID_API_KEY")
SENDGRID_SENDER_ADDRESS = userdata.get("SENDGRID_SENDER_ADDRESS")

### AlphaVantage API Setup

Obtain an [AlphaVantage API Key](https://www.alphavantage.co/support/#api-key). A normal key should be fine, but alternatively you can use one of the prof's "premium" keys.  

Set the API key as a notebook secret called `ALPHAVANTAGE_API_KEY`.

In [5]:
from google.colab import userdata

ALPHAVANTAGE_API_KEY = userdata.get("ALPHAVANTAGE_API_KEY")

### USD Formatter Function

In [6]:
# without docstring:
#def format_usd(my_price):
#    return f"${float(my_price):.2f}"

# with docstring:
def format_usd(my_price):
    """Formats a price as USD with dollar sign, rounded to two decimal places.

    Params:
        my_price (float or str): a price like 3.5

    Returns (str) like '$3.50'
    """
    return f"${float(my_price):.2f}"



Example invocations:

In [7]:
print(format_usd(3.5))

print(format_usd(123.555555))

result = format_usd(123.555555)
print(result)

$3.50
$123.56
$123.56


Example tests:

In [8]:
assert format_usd(3.5) == '$3.50'

assert format_usd(123.555555) == '$123.56'

result = format_usd(123.555555)
assert result == '$123.56'

## Demo

### Stock Selection

In [18]:
symbol = "SPOT" #@param ["AAPL", "MSFT", "GOOGL", "NFLX", "SPOT"]
print("SYMBOL:", symbol)

SYMBOL: SPOT


### Fetching Data (CSV)

In [19]:
from pandas import read_csv

request_url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={symbol}&apikey={ALPHAVANTAGE_API_KEY}&outputsize=full&datatype=csv"

df = read_csv(request_url)
print(df.columns)
print(len(df))
df.head()

Index(['timestamp', 'open', 'high', 'low', 'close', 'adjusted_close', 'volume',
       'dividend_amount', 'split_coefficient'],
      dtype='object')
1894


,timestamp,open,high,low,close,adjusted_close,volume,dividend_amount,split_coefficient
0,2025-10-13,677.20,694.9300,675.2650,691.75,691.75,1029698,0.0,1.0
1,2025-10-10,673.72,689.5499,670.0000,685.29,685.29,1377250,0.0,1.0
2,2025-10-09,673.70,675.0400,666.0001,673.66,673.66,1164659,0.0,1.0
3,2025-10-08,689.38,696.5000,673.5000,674.75,674.75,1431566,0.0,1.0
4,2025-10-07,679.98,683.0000,670.5000,680.44,680.44,1255718,0.0,1.0


In [20]:
# Challenge A

latest_close = df["adjusted_close"][0]
latest_date = df["timestamp"][0]

print("LATEST CLOSE:", f"{format_usd(latest_close)}")
print("LATEST DATE:", latest_date)

LATEST CLOSE: $691.75
LATEST DATE: 2025-10-13


### Dataviz

In [21]:
# Challenge C
#
# Plot a line chart of adjusted closing prices over time (all time).

from plotly.express import line
import plotly.io as pio

fig = line(x=df["timestamp"], y=df["adjusted_close"],
            title=f"Stock Prices ({symbol})",
           labels= {"x": "Date", "y": "Stock Price ($)"})
print(type(fig))
fig.show()

<class 'plotly.graph_objs._figure.Figure'>


### Email Report

#### Message Contents

In [22]:
import base64

from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail, Attachment, FileContent, FileName, FileType, Disposition, ContentId

# PREPARE MESSAGE

subject = f"Awesome Stocks Report for {symbol}"

html_content = f"""
    <h2>This is your Awesome Stocks Report!</h2>

    <p>The latest adjusted closing price for {symbol} is {format_usd(latest_close)} (as of {latest_date}).</p>

    <p>See attached files.</p>
"""

client = SendGridAPIClient(SENDGRID_API_KEY)
message = Mail(
    from_email=SENDGRID_SENDER_ADDRESS,
    to_emails=SENDGRID_SENDER_ADDRESS,
    subject=subject,
    html_content=html_content
)

#### CSV Attachment

In [23]:
df.to_csv(index=False)

'timestamp,open,high,low,close,adjusted_close,volume,dividend_amount,split_coefficient\n2025-10-13,677.2,694.93,675.265,691.75,691.75,1029698,0.0,1.0\n2025-10-10,673.72,689.5499,670.0,685.29,685.29,1377250,0.0,1.0\n2025-10-09,673.7,675.04,666.0001,673.66,673.66,1164659,0.0,1.0\n2025-10-08,689.38,696.5,673.5,674.75,674.75,1431566,0.0,1.0\n2025-10-07,679.98,683.0,670.5,680.44,680.44,1255718,0.0,1.0\n2025-10-06,684.27,698.85,675.45,680.28,680.28,1289524,0.0,1.0\n2025-10-03,706.44,708.55,680.24,680.5,680.5,1731800,0.0,1.0\n2025-10-02,700.0,716.86,698.49,707.3,707.3,1347175,0.0,1.0\n2025-10-01,699.5,721.67,697.78,700.68,700.68,2172340,0.0,1.0\n2025-09-30,705.26,711.0,677.99,698.0,698.0,4130566,0.0,1.0\n2025-09-29,726.91,740.0,720.275,728.47,728.47,965761,0.0,1.0\n2025-09-26,712.62,716.53,704.44,716.53,716.53,1198582,0.0,1.0\n2025-09-25,698.76,719.75,695.97,712.62,712.62,1238285,0.0,1.0\n2025-09-24,715.35,719.74,699.32,708.65,708.65,1405694,0.0,1.0\n2025-09-23,736.14,740.0,716.85,718.94,718.

In [24]:
# PREPARE ATTACHMENT (CSV FILE)

encoded_csv = base64.b64encode(df.to_csv(index=False).encode()).decode()

csv_attachment_filename = f"{symbol.lower()}_prices_{latest_date.replace('-','')}.csv"
print(csv_attachment_filename)

message.attachment = Attachment(
    file_content = FileContent(encoded_csv),
    file_type = FileType('text/csv'),
    file_name = FileName(csv_attachment_filename),
    disposition = Disposition('attachment'),
    content_id = ContentId('Attachment 1')
)


spot_prices_20251013.csv


#### PNG Attachment

In [25]:
# PREPARE ATTACHMENT (IMAGE FILE)

img_bytes = fig.to_image(format="png") # NOTE: image export requires the "kaleido" package
encoded_img = base64.b64encode(img_bytes).decode()

img_attachment_filepath = f"{symbol.lower()}_prices_{latest_date.replace('-','')}.png"
print(img_attachment_filepath)

message.attachment = Attachment(
    file_content = FileContent(encoded_img),
    file_type = FileType('image/png'),
    file_name = FileName(img_attachment_filepath),
    disposition = Disposition('attachment'),
    content_id = ContentId('Attachment 2')
)

spot_prices_20251013.png


#### Sending the Message

In [26]:
#
# SEND MESSAGE
#

response = client.send(message)
print(response.status_code)

202


Here is a screenshot of the email:

![TODO](...)